In [0]:

from pyspark.sql.functions import *
from pyspark.sql import Window
from delta.tables import DeltaTable

bronze_table = "data_modelling_demo_catalog.bronze.scd_2_output"
silver_table = "data_modelling_demo_catalog.silver.scd_2_output"

dbutils.widgets.text("insert_timestamp", "", "Insert Timestamp (YYYY-MM-DD HH:MM:SS)")
insert_timestamp = dbutils.widgets.get("insert_timestamp")

# 1. Read Bronze
if not insert_timestamp:
    df = spark.read.table(bronze_table)
else:
    df = (
        spark.read.table(bronze_table)
        .filter(
            col("ingestion_time") >
            to_timestamp(lit(insert_timestamp))
        )
    )

# 2. Transform the data
transformed_df = (
    df.withColumn("first_name", initcap(trim(col("first_name"))))
      .withColumn("email", lower(trim(col("email"))))
      .withColumn(
          "registration_date",
          to_date(col("registration_date"), "dd-MMM-yyyy")
      )
)

# 3. Data quality checks
valid_conditions = (
    col("customer_id").isNotNull()
    & col("first_name").isNotNull()
    & (trim(col("first_name")) != "")
    & col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    & col("registration_date").isNotNull()
)

valid_df = transformed_df.filter(valid_conditions)
invalid_df = transformed_df.filter(~valid_conditions)

print("Invalid records:", invalid_df.count())

# 4. Keep the latest incoming record per customer
window_spec = Window.partitionBy("customer_id").orderBy(
    col("ingestion_time").desc()
)

incoming_df = (
    valid_df
    .withColumn("_row_num", row_number().over(window_spec))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
    .drop("processed_time", "valid_from", "valid_to", "is_current")
)

# Use one timestamp consistently for this processing batch
batch_time = spark.sql(
    "SELECT current_timestamp() AS ts"
).first()["ts"]

# 5. First run: create the SCD2 table
if not spark.catalog.tableExists(silver_table):

    initial_df = (
        incoming_df
        .withColumn("valid_from", lit(batch_time).cast("timestamp"))
        .withColumn("valid_to", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
        .withColumn("processed_time", lit(batch_time).cast("timestamp"))
    )

    (
        initial_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

    print("SCD2 initial load completed.")

else:

    silver_dt = DeltaTable.forName(spark, silver_table)

    # Read the currently active versions
    current_df = (
        spark.read.table(silver_table)
        .filter(col("is_current") == True)
        .select(
            "customer_id",
            "first_name",
            "email",
            "city",
            "registration_date"
        )
    )

    # 6. Find new customers and customers whose tracked
    # attributes have changed.
    joined_df = incoming_df.alias("s").join(
        current_df.alias("t"),
        on="customer_id",
        how="left"
    )

    is_new = col("t.customer_id").isNull()

    has_changed = (
        ~col("s.first_name").eqNullSafe(col("t.first_name"))
        | ~col("s.email").eqNullSafe(col("t.email"))
        | ~col("s.city").eqNullSafe(col("t.city"))
        | ~col("s.registration_date").eqNullSafe(
            col("t.registration_date")
        )
    )

    changed_df = (
        joined_df
        .filter(is_new | has_changed)
        .select("s.*")
    )

    # 7. Expire the old version for changed customers.
    changed_ids = changed_df.select("customer_id").distinct()

    rows_to_expire = (
        changed_ids
        .withColumn("_expire", lit(True))
    )

    (
        silver_dt.alias("t")
        .merge(
            rows_to_expire.alias("s"),
            "t.customer_id = s.customer_id AND t.is_current = true"
        )
        .whenMatchedUpdate(
            set={
                "valid_to": lit(batch_time),
                "is_current": lit(False),
                "processed_time": lit(batch_time)
            }
        )
        .execute()
    )

    # 8. Insert the new version of every new or changed customer.
    new_versions = (
        changed_df
        .withColumn("valid_from", lit(batch_time).cast("timestamp"))
        .withColumn("valid_to", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
        .withColumn("processed_time", lit(batch_time).cast("timestamp"))
    )

    (
        new_versions.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    print("SCD2 incremental load completed.")
    print("New or changed customer versions:", changed_df.count())


In [0]:
%sql
select * from data_modelling_demo_catalog.silver.scd_2_output;
